In [1]:
import bw2data, bw2io
import bw2calc
import os
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from scipy.integrate import cumtrapz

In [2]:
from utils_pGWP.pGWP_V2 import *

In [3]:
bw2data.projects.set_current('iveo_v1')
mybio = bw2data.Database("biosphere3")
#initProject("iveo_v1")

### A1. prepare empty ds for assigning dLCIA score once its prepared, same ds as in other modules

In [4]:
v_lifetime = 20 
# Define dimensions
IC = ['IPCC2021_default', 'pGWP100', 'premise_GWP']
SSP = ['ssp119', 'ssp245', 'ssp585']
v_year = [2030,2040,2050]
tech_list = ['dummy_tech']
v_module = ['V2_oper_EF']
# Create a new xarray dataset with dimensions
cord = {'SSP': SSP, 'v_year': v_year, 'v_module': v_module, 'tech_list':tech_list,  'Impact_category': IC}
ds = xr.Dataset(coords=cord)

#   data array with fake data
fakedd = np.full((len(SSP), len(v_year), len(v_module), len(tech_list), len(IC)), np.nan)    
value = xr.DataArray(fakedd, dims=('SSP', 'v_year', 'v_module', 'tech_list', 'Impact_category'), coords=cord )
 
# Assign the data variable to the dataset
ds = ds.assign(value = value)
ds

<xarray.Dataset>
Dimensions:          (SSP: 3, v_year: 3, v_module: 1, tech_list: 1,
                      Impact_category: 3)
Coordinates:
  * SSP              (SSP) <U6 'ssp119' 'ssp245' 'ssp585'
  * v_year           (v_year) int64 2030 2040 2050
  * v_module         (v_module) <U10 'V2_oper_EF'
  * tech_list        (tech_list) <U10 'dummy_tech'
  * Impact_category  (Impact_category) <U16 'IPCC2021_default' ... 'premise_GWP'
Data variables:
    value            (SSP, v_year, v_module, tech_list, Impact_category) float64 ...

### A.2 get yearly LCI 
#### if you change annual km, change notebook in "prep_V2_exh_emis" folder

In [5]:
MYs = [2030 , 2040,2050 ]
lci_y = 20

In [6]:
LCI_dt_allY = []
for vy in MYs: 
    lci_file = os.path.join("../notebook_veh_wh_dLCIA/ICEV_dpLCA/prep_V2_exh_emis", "icev_d_ef_lci.xlsx")
    LCI_dt = pd.read_excel(lci_file, sheet_name = 'v_y' + str(vy)) 
    LCI_dt.head()
    LCI_dt_allY.append(LCI_dt)

In [7]:
LCI_dt_dict = {
               "vy2030":  LCI_dt_allY[0]  ,
               "vy2040":  LCI_dt_allY[1]  ,
               "vy2050":  LCI_dt_allY[2]  ,
              }

In [8]:
LCI_dt_dict.get('vy2030').head()

,year,CO2,CH4,N2O
0,1,45828,1.881,1.4022
1,2,45828,1.881,1.4022
2,3,45828,1.881,1.4022
3,4,45828,1.881,1.4022
4,5,45828,1.881,1.4022


### see what CFs are used for each SSP/MY: 
IPCC default has same as premise_gwp for emission flows, but out pGWP100 has different CFs prepared 

In [9]:
print(load_bw2mm_map_cf_for_CFdict (default_ipcc, map_majorghg_cfs )) 

print(load_bw2mm_map_cf_for_CFdict (premise_gwp, map_majorghg_cfs )) 

print(load_bw2mm_map_cf_for_CFdict (('IPCC 2021 - dpCFsSSP119_MY2030 - year100', 'climate change', 'pGWP100')
, map_majorghg_cfs )) 

print(load_bw2mm_map_cf_for_CFdict (('IPCC 2021 - dpCFsSSP119_MY2050 - year100', 'climate change', 'pGWP100')
, map_majorghg_cfs )) 

print(load_bw2mm_map_cf_for_CFdict (('IPCC 2021 - dpCFsSSP585_MY2050 - year100', 'climate change', 'pGWP100')
, map_majorghg_cfs )) 

{'CO2': ['d6235194-e4e6-4548-bfa3-ac095131aef4', 1], 'CH4': ['7f0ba7c9-341e-413d-80f6-8753727d65d1', 29.8], 'N2O': ['473826ae-125a-4b02-8c8e-c84322491d80', 273]}
{'CO2': ['d6235194-e4e6-4548-bfa3-ac095131aef4', 1], 'CH4': ['7f0ba7c9-341e-413d-80f6-8753727d65d1', 29.8], 'N2O': ['473826ae-125a-4b02-8c8e-c84322491d80', 273]}
{'CO2': ['d6235194-e4e6-4548-bfa3-ac095131aef4', 1], 'CH4': ['7f0ba7c9-341e-413d-80f6-8753727d65d1', 24.85550556235459], 'N2O': ['473826ae-125a-4b02-8c8e-c84322491d80', 191.5733076869126]}
{'CO2': ['d6235194-e4e6-4548-bfa3-ac095131aef4', 1], 'CH4': ['7f0ba7c9-341e-413d-80f6-8753727d65d1', 26.98650978777478], 'N2O': ['473826ae-125a-4b02-8c8e-c84322491d80', 180.3885394672563]}
{'CO2': ['d6235194-e4e6-4548-bfa3-ac095131aef4', 1], 'CH4': ['7f0ba7c9-341e-413d-80f6-8753727d65d1', 25.36191049861465], 'N2O': ['473826ae-125a-4b02-8c8e-c84322491d80', 231.7033626172096]}


## C. calc.  assign CFs directly, overwrite ds directly in the loop

#### get lci_df for each SSP/MY, then: 
1. using pGWP100 per SSP[x], MY[t] for the new pGWP100 method, or
2. using default_ipcc or premise_gwp, they've the same static GWP100 value


In [10]:
for sp in ds.SSP.values:
    for my in ds.v_year.values: 
        keytoget = 'vy'+str(my)
        lci_df = LCI_dt_dict.get(keytoget)
    
        for impact in  ds.Impact_category.values: 
            if impact == "pGWP100": 
                pgwpmm = get_my_pGWP100(ssp = sp.capitalize(), my  = my)
                #print(pgwpmm)
                cf_touse = load_bw2mm_map_cf_for_CFdict(pgwpmm, map_majorghg_cfs)
                co2e = calc_co2e_majorghg_v2(lci_df, cf_touse)
                #print(f" ~~~~~~~~~~~ for {sp}, v_year{my}, total CO2e kg is {co2e}")
            
            else: # same CFs for IPCC and premise_gwp for emission flows  
                mm = default_ipcc # always same IPCC CF 
                cf_touse2 = load_bw2mm_map_cf_for_CFdict(mm, map_majorghg_cfs)
                co2e = calc_co2e_majorghg_v2(lci_df, cf_touse2)

            ds['value'].loc[{'SSP': sp, 'v_year': my, 'v_module':'V2_oper_EF', 'tech_list':'dummy_tech', 'Impact_category':impact} ]  = co2e

            

for CO2, lifecycle emission kg are: 916560.0, using CF of ['d6235194-e4e6-4548-bfa3-ac095131aef4', 1] 
for CH4, lifecycle emission kg are: 37.62, using CF of ['7f0ba7c9-341e-413d-80f6-8753727d65d1', 29.8] 
for N2O, lifecycle emission kg are: 28.044, using CF of ['473826ae-125a-4b02-8c8e-c84322491d80', 273] 
for CO2, lifecycle emission kg are: 916560.0, using CF of ['d6235194-e4e6-4548-bfa3-ac095131aef4', 1] 
for CH4, lifecycle emission kg are: 37.62, using CF of ['7f0ba7c9-341e-413d-80f6-8753727d65d1', 24.85550556235459] 
for N2O, lifecycle emission kg are: 28.044, using CF of ['473826ae-125a-4b02-8c8e-c84322491d80', 191.5733076869126] 
for CO2, lifecycle emission kg are: 916560.0, using CF of ['d6235194-e4e6-4548-bfa3-ac095131aef4', 1] 
for CH4, lifecycle emission kg are: 37.62, using CF of ['7f0ba7c9-341e-413d-80f6-8753727d65d1', 29.8] 
for N2O, lifecycle emission kg are: 28.044, using CF of ['473826ae-125a-4b02-8c8e-c84322491d80', 273] 
for CO2, lifecycle emission kg are: 870731.999

In [11]:
ds

<xarray.Dataset>
Dimensions:          (SSP: 3, v_year: 3, v_module: 1, tech_list: 1,
                      Impact_category: 3)
Coordinates:
  * SSP              (SSP) <U6 'ssp119' 'ssp245' 'ssp585'
  * v_year           (v_year) int64 2030 2040 2050
  * v_module         (v_module) <U10 'V2_oper_EF'
  * tech_list        (tech_list) <U10 'dummy_tech'
  * Impact_category  (Impact_category) <U16 'IPCC2021_default' ... 'premise_GWP'
Data variables:
    value            (SSP, v_year, v_module, tech_list, Impact_category) float64 ...

In [12]:
ds.sel(SSP = 'ssp119', v_year = 2050, tech_list = 'dummy_tech', v_module = "V2_oper_EF")['value']

<xarray.DataArray 'value' (Impact_category: 3)>
array([835116.72192   , 832677.22895088, 835116.72192   ])
Coordinates:
    SSP              <U6 'ssp119'
    v_year           int64 2050
    v_module         <U10 'V2_oper_EF'
    tech_list        <U10 'dummy_tech'
  * Impact_category  (Impact_category) <U16 'IPCC2021_default' ... 'premise_GWP'

In [13]:
ds.to_netcdf('pGWP_module_output/V2_ICEVdGT_pGPW100.nc')

In [14]:
df = ds.to_dataframe() #.reset_index()
df

value
SSP    v_year v_module   tech_list  Impact_category                
ssp119 2030   V2_oper_EF dummy_tech IPCC2021_default  925337.088000
                                    pGWP100           922867.545960
                                    premise_GWP       925337.088000
       2040   V2_oper_EF dummy_tech IPCC2021_default  879070.233600
                                    pGWP100           876671.642052
                                    premise_GWP       879070.233600
       2050   V2_oper_EF dummy_tech IPCC2021_default  835116.721920
                                    pGWP100           832677.228951
                                    premise_GWP       835116.721920
ssp245 2030   V2_oper_EF dummy_tech IPCC2021_default  925337.088000
                                    pGWP100           922986.111396
                                    premise_GWP       925337.088000
       2040   V2_oper_EF dummy_tech IPCC2021_default  879070.233600
                                    pGWP100           877081.440251
                                    premise_GWP       879070.233600
       2050   V2_oper_EF dummy_tech IPCC2021_default  835116.721920
                                    pGWP100           833420.807008
                                    premise_GWP       835116.721920
ssp585 2030   V2_oper_EF dummy_tech IPCC2021_default  925337.088000
                                    pGWP100           923016.797792
                                    premise_GWP       925337.088000
       2040   V2_oper_EF dummy_tech IPCC2021_default  879070.233600
                                    pGWP100           877256.718713
                                    premise_GWP       879070.233600
       2050   V2_oper_EF dummy_tech IPCC2021_default  835116.721920
                                    pGWP100           833920.833767
                                    premise_GWP       835116.721920